---
title: Building a Simple Teaching Agent
authors:
  - name: Huan Zhang
---

# Building a simple teaching agent

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/music-llm-tutorial/music-llm-tutorial.github.io/blob/main/book/performing/agentic_demo.ipynb)

The previous three sections described the ingredients: MIR tools, learner memory, and pedagogy as the policy. This notebook assembles a minimal teaching agent (TA) from those ingredients: a real transcription model, a real score comparison, three enrolled learners, and an agent loop in about forty lines, with no framework.

The setting is a small practice studio. You log in as one of three learners, hand the TA that learner's newest take of the weekly exercise, and watch it transcribe the audio, grade it against the score, deliver feedback in the learner's own style, and write back what it learned for next week.

Three scope decisions keep the notebook self-contained:

- **The recordings are synthesized.** Each learner's take is rendered from the score with that learner's mistakes injected: a dragging run for one, a creeping tempo for another. Synthesis keeps the notebook reproducible and free of real children's recordings, and gives us ground truth to check the pipeline against. Any wav dropped into the same folder goes through the identical path.
- **The transcription is real.** Takes go through basic-pitch, a lightweight instrument-agnostic note transcription model that runs on CPU {cite}`bittner_basicpitch_2022`, then through a deliberately naive matcher against the score. The full-strength alternatives are surveyed in [](./agentic_tools.md).
- **The learner database is fixed.** Three profiles, no sign-up. This is enough to demonstrate memory and personalisation, the two behaviours that separate a teaching agent from a chatbot.

## What a domain-specific agent is

Strip away framework vocabulary and a tool-using agent is four artefacts {cite}`chu_llm_agents_education_2025`:

1. **A model**, reached through an API that supports tool calling.
2. **A system prompt** that encodes the domain policy, here the pedagogy of [](./agentic_pedagogy.md), plus the context the model must always have.
3. **Tools**: JSON schemas describing functions the model may request, plus your implementations of those functions.
4. **A loop** that sends the conversation, executes whatever tools the model requested, appends the results, and repeats until the model stops requesting.

Everything a framework advertises on top (planning, memory, orchestration) is built from these four.

We use the Anthropic API below. OpenAI and Gemini expose the same protocol under the name *function calling*, and porting this notebook to either is a field-renaming exercise.

In [ ]:
%pip install -q anthropic basic-pitch soundfile numpy

basic-pitch downloads its model weights on first use; if pip cannot resolve it on a very new Python version, use Python 3.11. Running the two TA sessions below makes a handful of model calls, a few cents in total. In Colab, store your key under the key icon in the left sidebar or paste it when prompted.

In [ ]:
import os, getpass

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API key: ")

## The studio database

A first instinct for "give the model context" is retrieval-augmented generation: embed everything, retrieve by similarity. For learner state, that is the wrong shape. A TA's context is small, structured, and keyed by identity: one profile, one memory list, a handful of recordings, all belonging to the logged-in user. The natural access pattern is a lookup keyed by the login, not a nearest-neighbour search over documents. RAG earns its place one layer out, over pedagogical knowledge such as method books and technique treatises, where the corpus is large and the query is fuzzy; that layer is orthogonal to this notebook.

Each learner file holds the four kinds of state that [](./agentic_memory.md) argued a TA needs: identity and level, style preferences, hard constraints from the human teacher, and dated memory notes from previous sessions.

In [1]:
import json, pathlib, shutil

DB = pathlib.Path("studio_db")
if DB.exists():
    shutil.rmtree(DB)
(DB / "learners").mkdir(parents=True)
(DB / "recordings").mkdir()

learners = {
    "amy": {
        "name": "Amy", "age": 10, "instrument": "piano",
        "level": "beginner, around ABRSM grade 2",
        "preferences": {
            "feedback_style": "imagery and analogy, short sentences, one idea at a time",
            "encouragement": "high",
        },
        "teacher_notes": (
            "Hands-together tempo capped at 80 bpm until the September lesson. "
            "No octave stretches, small hands."
        ),
        "active_repertoire": ["Burgm\u00fcller, Arabesque Op. 100 No. 2"],
        "memory": [
            {"date": "2026-08-10",
             "note": "Sixteenth runs in the weekly exercise uneven at speed, worse when "
                     "the line climbs. Assigned slow practice at 60 bpm."},
            {"date": "2026-08-17",
             "note": "Runs still drag in the second half of the exercise, slow practice "
                     "done inconsistently. The galloping-horse rhythm analogy landed well."},
        ],
    },
    "ben": {
        "name": "Ben", "age": 34, "instrument": "piano",
        "level": "intermediate, returned to the piano after ten years away",
        "preferences": {
            "feedback_style": "direct and technical, quantitative detail welcome, skip the praise",
            "encouragement": "low",
        },
        "teacher_notes": (
            "Self-directed, no regular teacher. "
            "Flag anything that needs human eyes rather than guessing."
        ),
        "active_repertoire": ["Bach, Invention No. 1 in C major, BWV 772"],
        "memory": [
            {"date": "2026-08-12",
             "note": "Prefers metronome-based drills. Asked for honesty over encouragement."},
        ],
    },
    "chloe": {
        "name": "Chloe", "age": 20, "instrument": "piano",
        "level": "advanced, second-year conservatory",
        "preferences": {
            "feedback_style": "detailed musical discussion, comfortable with theory terminology",
            "encouragement": "moderate",
        },
        "teacher_notes": (
            "Recital on 3 October. Interpretation decisions belong to her professor; "
            "the agent sticks to accuracy and consistency."
        ),
        "active_repertoire": ["Chopin, Nocturne Op. 9 No. 2"],
        "memory": [
            {"date": "2026-08-15",
             "note": "Run-through mostly solid. Records in her flat, room acoustics are poor."},
        ],
    },
}

for uid, profile in learners.items():
    (DB / "learners" / f"{uid}.json").write_text(
        json.dumps(profile, indent=2, ensure_ascii=False))

current_user = None

def login(user_id: str) -> None:
    global current_user
    current_user = user_id
    profile = json.loads((DB / "learners" / f"{user_id}.json").read_text())
    print(f"Logged in as {profile['name']} ({profile['level']})")

## The score and the takes

The studio's weekly assignment is a four-bar sixteenth-note exercise at 100 bpm, encoded as a list of (onset, duration, MIDI pitch) triples. Real systems hold MusicXML and align it to audio; a plain note list is the smallest representation the comparison below needs.

Each learner's take is synthesized from that score with their mistakes injected, then written to disk as a wav like any uploaded recording. Because we injected the mistakes ourselves, we know exactly what the pipeline *should* find.

In [2]:
import numpy as np, soundfile as sf
from IPython.display import Audio, display

SR, BPM = 22050, 100
SIXTEENTH = 60 / BPM / 4
MELODY = [60,62,64,65,67,65,64,62, 60,64,67,72,71,67,64,62,
          64,65,67,69,71,69,67,65, 64,67,71,76,74,71,67,64]
SCORE = [(i * SIXTEENTH, SIXTEENTH * 0.95, p) for i, p in enumerate(MELODY)]

def synth(notes):
    total = max(o + d for o, d, _ in notes) + 0.5
    audio = np.zeros(int(total * SR))
    for onset, dur, midi in notes:
        f = 440 * 2 ** ((midi - 69) / 12)
        t = np.arange(int(dur * SR)) / SR
        tone = sum((0.6 ** k) * np.sin(2 * np.pi * f * (k + 1) * t) for k in range(4))
        i = int(onset * SR)
        audio[i:i + len(tone)] += tone * np.exp(-3.0 * t) * 0.3
    return audio / (np.max(np.abs(audio)) + 1e-9)

def make_take(user):
    notes = []
    for i, (onset, dur, pitch) in enumerate(SCORE):
        if user == "amy":
            if 20 <= i <= 27: onset += 0.055 + 0.01 * (i - 20)   # second-half run drags
            if i == 11: pitch = 71                               # wrong note in bar 2
        if user == "ben":
            onset *= 0.985                                       # creeps ahead of the beat
            if i == 13: pitch = 68                               # one slip
        notes.append((onset, dur, pitch))
    audio = synth(notes)
    if user == "chloe":
        audio = audio + np.random.default_rng(0).normal(0, 0.15, len(audio))  # bad room
    sf.write(DB / "recordings" / f"{user}_take.wav", audio, SR)

for user in learners:
    make_take(user)

print("the exercise as written:")
display(Audio(synth(SCORE), rate=SR))
print("Amy's take:")
display(Audio(sf.read(DB / "recordings" / "amy_take.wav")[0], rate=SR))

the exercise as written:


Amy's take:


Three details are deliberate. Amy's drag sits in the second half, the same problem her memory notes record for two prior weeks, so a good TA should escalate strategy rather than repeat last week's advice. Ben plays every note yet is never quite with the beat, an error that only shows up in aggregate. Chloe's take went through a bad room; she waits in the exercises.

## Real tools: transcription and score comparison

A tool has two halves. The *schema* is what the model sees: a name, a natural-language description, and typed parameters. The *function* is what runs when the model requests that name. Descriptions are prompt engineering: the model decides when to call each tool purely from that text.

Two of the three tools below do real MIR work. `transcribe_recording` runs basic-pitch and returns raw note events. `check_against_score` matches those events to the score, nearest onset within 100 ms, and reports per bar: matched counts, pitch mismatches, missed notes, and mean onset deviation. The matcher is deliberately the dumbest thing that works; [](./agentic_tools.md) surveys what replaces it in earnest systems.

The model never touches the filesystem. `current_user` lives in Python, out of the model's reach, and every function filters by it; in a deployed product this same boundary carries authentication and child-safety controls.

In [3]:
import contextlib, io
from basic_pitch.inference import predict

_transcriptions = {}

def _played(recording_id):
    if recording_id not in _transcriptions:
        quiet = contextlib.redirect_stdout(io.StringIO())
        with quiet, contextlib.redirect_stderr(io.StringIO()):
            _, _, events = predict(str(DB / "recordings" / f"{recording_id}.wav"))
        _transcriptions[recording_id] = sorted(
            (round(float(s), 3), round(float(e - s), 3), int(p))
            for s, e, p, amp, bends in events)
    return _transcriptions[recording_id]

def list_recordings() -> str:
    return json.dumps([p.stem for p in
                       sorted((DB / "recordings").glob(f"{current_user}_*.wav"))])

def transcribe_recording(recording_id: str) -> str:
    if not recording_id.startswith(current_user):
        return json.dumps({"error": "recording belongs to another learner"})
    return json.dumps([{"onset_s": o, "duration_s": d, "midi_pitch": p}
                       for o, d, p in _played(recording_id)])

def check_against_score(recording_id: str) -> str:
    if not recording_id.startswith(current_user):
        return json.dumps({"error": "recording belongs to another learner"})
    played = [(o, p) for o, d, p in _played(recording_id)]
    used, bars = set(), []
    notes_per_bar, tol = 8, 0.10
    for bar in range(len(SCORE) // notes_per_bar):
        rows = []
        for j in range(bar * notes_per_bar, (bar + 1) * notes_per_bar):
            exp_onset, _, exp_pitch = SCORE[j]
            best = None
            for k, (onset, pitch) in enumerate(played):
                if k in used or abs(onset - exp_onset) > tol:
                    continue
                if best is None or abs(onset - exp_onset) < abs(played[best][0] - exp_onset):
                    best = k
            beat = 1 + (j % notes_per_bar) / 4
            if best is None:
                rows.append({"beat": beat, "expected_pitch": exp_pitch, "heard_pitch": None})
            else:
                used.add(best)
                onset, pitch = played[best]
                rows.append({"beat": beat, "expected_pitch": exp_pitch, "heard_pitch": pitch,
                             "onset_dev_ms": round((onset - exp_onset) * 1000)})
        devs = [r["onset_dev_ms"] for r in rows if r["heard_pitch"] is not None]
        bars.append({
            "bar": bar + 1,
            "notes_matched": f"{len(devs)}/{notes_per_bar}",
            "pitch_errors": [r for r in rows
                             if r["heard_pitch"] not in (None, r["expected_pitch"])],
            "missed_notes": [r for r in rows if r["heard_pitch"] is None],
            "mean_onset_dev_ms": round(sum(devs) / len(devs)) if devs else None,
        })
    return json.dumps({"tempo_bpm_assigned": BPM, "bars": bars,
                       "extra_unmatched_notes": len(played) - len(used)})

TOOL_FUNCTIONS = {f.__name__: f for f in
                  [list_recordings, transcribe_recording, check_against_score]}

TOOLS = [
    {
        "name": "list_recordings",
        "description": "List the logged-in learner's uploaded recordings by id.",
        "input_schema": {"type": "object", "properties": {}, "required": []},
    },
    {
        "name": "transcribe_recording",
        "description": "Run automatic music transcription (basic-pitch) on one recording "
                       "and return raw note events: onset in seconds, duration, MIDI pitch. "
                       "Use this to inspect raw evidence; prefer check_against_score for "
                       "graded feedback.",
        "input_schema": {"type": "object",
                         "properties": {"recording_id": {"type": "string"}},
                         "required": ["recording_id"]},
    },
    {
        "name": "check_against_score",
        "description": "Compare one recording against the assigned exercise score. Returns "
                       "a per-bar report: matched note counts, pitch mismatches, missed "
                       "notes, mean onset deviation in ms (positive means late), plus a "
                       "count of extra unmatched notes. Matching is nearest-onset within "
                       "100 ms, so isolated single-note anomalies may be transcription "
                       "artefacts rather than playing errors.",
        "input_schema": {"type": "object",
                         "properties": {"recording_id": {"type": "string"}},
                         "required": ["recording_id"]},
    },
]

scikit-learn version 1.9.0 is not supported. Minimum required version: 0.17. Maximum required version: 1.5.1. Disabling scikit-learn conversion API.


/Users/huanz/.cache/uv/archive-v0/lVLn-tunn8KOAERFqJYjL/lib/python3.11/site-packages/resampy/filters.py:50: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Before wiring these into an agent, look at what they actually return. This is the entirety of what the model will know about the playing:

In [4]:
login("amy")
report = json.loads(check_against_score("amy_take"))
for bar in report["bars"]:
    print(bar)
print("extra unmatched notes:", report["extra_unmatched_notes"])

Logged in as Amy (beginner, around ABRSM grade 2)


{'bar': 1, 'notes_matched': '7/8', 'pitch_errors': [], 'missed_notes': [{'beat': 1.0, 'expected_pitch': 60, 'heard_pitch': None}], 'mean_onset_dev_ms': -10}
{'bar': 2, 'notes_matched': '8/8', 'pitch_errors': [{'beat': 1.75, 'expected_pitch': 72, 'heard_pitch': 71, 'onset_dev_ms': -1}], 'missed_notes': [], 'mean_onset_dev_ms': -6}
{'bar': 3, 'notes_matched': '8/8', 'pitch_errors': [], 'missed_notes': [], 'mean_onset_dev_ms': 25}
{'bar': 4, 'notes_matched': '7/8', 'pitch_errors': [], 'missed_notes': [{'beat': 1.75, 'expected_pitch': 76, 'heard_pitch': None}], 'mean_onset_dev_ms': 28}
extra unmatched notes: 2


Read the report against what we injected. The wrong note in bar 2 is caught. Bars 3 and 4 show the drag as a rising mean onset deviation while bars 1 and 2 sit near zero. But the report also contains things Amy never played: the first note is missed (transcription regularly loses the very first onset) and a couple of extra notes are harmonic ghosts. Real MIR evidence arrives noisy, which is exactly why the pedagogy below tells the model to weight consistent patterns over isolated anomalies, the evidence-with-uncertainty point of [](./agentic_tools.md).

## Structure, not tools: profile and memory

Neither the learner profile nor memory writing appears in the tool list, and the omission is deliberate. A tool is an action the model may *choose*. Knowing the student is not a choice, it is a precondition of every reply, so the profile and memory notes are injected into the system prompt when the learner logs in. That is the context engineering: the model starts every turn already knowing who it teaches.

Remembering is not a choice either. If the memory write were a tool, some sessions would end unrecorded whenever the model decided not to call it. So the harness owns it: `end_session` makes one extra model call asking for a consolidated note, then appends it to the learner file itself. This is the assess, update state, act loop of ScaffoldLM {cite}`li_scaffoldlm_2026` with the update step guaranteed by code rather than requested by prompt.

In [5]:
PEDAGOGY = """\
You are a between-lessons teaching agent (TA) for one instrumental learner. You support \
their human teacher; you do not replace them.

Rules, in priority order:

1. Match the learner's age, level, and preferred feedback style, all given in the profile \
below. Constraints in teacher_notes are hard limits.
2. Tool output is evidence, not truth. Transcription misses notes (especially the first \
onset) and invents harmonic ghosts; weight consistent patterns over isolated one-note \
anomalies. If the report looks unreliable overall, say so and ask for another take.
3. Give formative feedback: tied to specific bars, about the playing rather than the \
person. Lead with the single most important issue and mention at most one more. Never \
recite the full report.
4. Assign one concrete drill for the coming week, sized just beyond what the memory notes \
say the learner already manages. If the same issue recurs across sessions, change strategy \
instead of repeating the old assignment.
"""

SYSTEM_PROMPT = None

def login(user_id: str) -> None:
    global current_user, SYSTEM_PROMPT
    current_user = user_id
    profile = json.loads((DB / "learners" / f"{user_id}.json").read_text())
    SYSTEM_PROMPT = (PEDAGOGY
        + "\nThe learner, their teacher's constraints, and your memory of past sessions:\n\n"
        + json.dumps(profile, indent=2, ensure_ascii=False))
    print(f"Logged in as {profile['name']} ({profile['level']})")

## The loop and the session boundary

This is the entire mechanism. The API is stateless, so `messages` carries the whole conversation every time. When the model wants a tool it stops with `stop_reason == "tool_use"`; we run the matching functions, append the results as a user message, and call again. When it answers in plain text, the turn is over. `end_session` closes the loop from the memory side.

Every agent framework you will meet is a wrapper around this while loop.

In [ ]:
import anthropic
from datetime import date
from IPython.display import Markdown

client = anthropic.Anthropic()
MODEL = "claude-opus-5"

def run_turn(messages: list) -> str:
    while True:
        response = client.messages.create(
            model=MODEL,
            max_tokens=4096,
            system=SYSTEM_PROMPT,
            tools=TOOLS,
            messages=messages,
        )
        messages.append({"role": "assistant", "content": response.content})
        if response.stop_reason != "tool_use":
            return next((b.text for b in response.content if b.type == "text"), "")
        results = []
        for block in response.content:
            if block.type == "tool_use":
                print(f"  tool call: {block.name}({json.dumps(block.input)})")
                output = TOOL_FUNCTIONS[block.name](**block.input)
                results.append({"type": "tool_result",
                                "tool_use_id": block.id,
                                "content": output})
        messages.append({"role": "user", "content": results})

def ask_ta(user_text: str, messages: list) -> None:
    messages.append({"role": "user", "content": user_text})
    display(Markdown(run_turn(messages)))

def end_session(messages: list) -> None:
    response = client.messages.create(
        model=MODEL, max_tokens=1024, system=SYSTEM_PROMPT,
        messages=messages + [{"role": "user", "content":
            "The session is over. Write one consolidated memory note for the next "
            "session: what you observed, what you assigned, anything worth carrying "
            "forward. Reply with the note text only."}],
    )
    note = next(b.text for b in response.content if b.type == "text")
    path = DB / "learners" / f"{current_user}.json"
    profile = json.loads(path.read_text())
    profile["memory"].append({"date": date.today().isoformat(), "note": note})
    path.write_text(json.dumps(profile, indent=2, ensure_ascii=False))
    print(f"memory saved for {current_user}: {note}")

## Session one: Amy

Amy is ten, plays at grade 2 level, and has dragged the same runs for three weeks. Nothing in the message below tells the agent any of that; the profile arrived through `login`, the evidence arrives through the tools.

In [ ]:
login("amy")
amy_session = []
ask_ta("I recorded the exercise this morning, can you check how it went?", amy_session)

Watch the printed tool calls: the agent chose them itself, nothing in our loop scripts the order. In our runs it checks the take against the score, leads with the second-half drag rather than reciting all four bars, speaks in imagery because the profile asks for it, keeps any tempo advice under the 80 bpm cap because teacher_notes is a hard limit, and does not scold Amy for the missed first note, which rule 2 correctly treats as a transcription artefact. Because the memory notes show week three of the same drag, it should also change tactics rather than repeat the slow-practice assignment. Model outputs vary run to run, so your transcript will read differently.

Follow-ups work without re-analysis, since the conversation is stateful within a session:

In [ ]:
ask_ta("It always goes wrong when I try to play it fast. What should I do this week?", amy_session)

Closing the session writes the note the next session will read:

In [ ]:
end_session(amy_session)

In [ ]:
for entry in json.loads((DB / "learners" / "amy.json").read_text())["memory"]:
    print(f"{entry['date']}  {entry['note']}")

The last entry was written by the model, but its persistence was guaranteed by the harness. Next week's `login` folds it into the system prompt, which is the whole difference between a reactive session bot and a longitudinal teaching agent in the sense of [](./agentic_memory.md). Note the compression choice: one curated note per session rather than the raw transcript, the inspectable-memory pattern from DeepTutor {cite}`deeptutor_2026`.

## Same architecture, different learner

Ben is an adult returner who asked for numbers and no praise. His take contains a different kind of error: every note is present, but he creeps ahead of the beat by a few percent, which no single note reveals and the per-bar deviation trend does. The code path is identical; only the injected profile and the audio changed.

In [ ]:
login("ben")
ben_session = []
ask_ta("Ran the exercise a few times and uploaded the best one. Numbers, please.", ben_session)
end_session(ben_session)

The register should shift audibly: millisecond deviations and per-bar trends for Ben, galloping horses for Amy, from the same system prompt scaffold and the same loop. Personalisation lives in the data, not in the code, which is why deployed practice products spend their engineering on the profile and memory schema rather than on prompt variants per user.

## Exercises

1. **Chloe's abstention test.** Her take went through a noisy room, and the transcription degrades accordingly. Log in as her, ask for feedback, and check whether the TA follows rule 2: refuse to grade unreliable evidence and ask for a cleaner take. If it confidently invents a diagnosis instead, tighten the rule and rerun; this is what prompt iteration on an agent actually looks like.
2. **A real human.** Record yourself playing the four bars on any instrument, drop the wav into `studio_db/recordings/` as `yourname_take.wav`, add a profile, and see how the pipeline copes with room noise, rubato, and breath.
3. **A better matcher.** Nearest-onset within 100 ms mislabels anything that drifts more than a sixteenth. Swap in note matching from Parangonar {cite}`peter_asap_alignments_2023` and compare the reports on the same takes.
4. **Polyphony.** basic-pitch transcribes chords; extend `SCORE` and the matcher to a hands-together exercise and find where naive matching breaks first.
5. **Forgetting.** After five memory entries, have `end_session` consolidate them into one summary entry. Decide what survives, guided by [](./agentic_memory.md).

## What made it domain-specific

The agent code contains no music, but the agent is steeped in it: the model's pretrained prior supplies the concepts and vocabulary, the system prompt injects the doctrine and the learner, and the tools return the evidence. Where the *fit* to the domain lives is more specific, and practitioner discussions of production agents converge on the same five places:

- **Data**: the learner schema, the score representation, and the recordings. The TA is exactly as good as the state you keep; most of the engineering in a real product goes here.
- **Tools**: the three functions, which decide what the model can observe. The MIR stack of [](./agentic_tools.md) enters the agent only through this surface.
- **Context engineering**: what reaches the prompt each turn. The profile arrives via `login`, evidence arrives curated through tool returns rather than as raw audio frames, and `end_session` controls what gets written back.
- **Tone**: the register shift between Amy and Ben, driven by profile preferences rather than separate prompts.
- **Guardrails**: `teacher_notes` as hard limits in the prompt, the login boundary enforced in code where the model cannot reach it, and abstention on unreliable evidence.

The same discussions rank a sixth element above fine-tuning: a verification loop against domain ground truth. The transcription and comparison stack plays that role here. The TA never trusts its own listening; it checks the take against note-level evidence, for the same reason a legal agent should verify a claim against the statute rather than recall it. Fine-tuning enters last, once knowledge access and verification are solved. The piece this notebook genuinely lacks is an evaluation harness that scores teaching quality across simulated learners; [](./agentic_pedagogy.md) closed on the observation that longitudinal benchmarks for music practice barely exist.

## How much can we trust the TA's domain knowledge

Everything the TA said about rhythm, fingering, and practice strategy came from its pretrained prior, uncited and unverified. How far that prior extends is measurable. MusicTheoryBench, introduced with ChatMusician, tests college-level written theory in a knowledge split and a reasoning split {cite}`yuan_chatmusician_2024`. GPT-4 answered a little under sixty percent of the knowledge split and stayed near chance on multi-step reasoning, far below trained musicians on both. And neither split measures pedagogical judgement, the capability this chapter actually needs: knowing the rules of counterpoint is not knowing how to fix a ten-year-old's rushed run.

The practical stance mirrors expert domains such as law and medicine: trust the prior for concepts and vocabulary, ground everything that could be looked up. In increasing order of cost:

- **Injected doctrine**: rubrics, grade syllabi, and methodology playbooks placed in the system prompt or loaded on demand. The four rules above are this layer in miniature.
- **Retrieval over a pedagogy corpus**: the canonical treatises are public domain and number in the dozens, so a TA can quote its sources with simple retrieval and no vector infrastructure. Clinical and legal agents live on this pattern at far larger scale, because a retrieved claim carries a citation the user can check and a recalled claim does not.
- **Fine-tuning and continued pretraining**: reshaping the model itself on domain text, as ChatMusician does with theory texts and ABC notation {cite}`yuan_chatmusician_2024`. It buys domain fluency and style more reliably than facts, and a tuned model still cannot cite its sources, so production agents reach for it only after the cheaper layers are exhausted.

Where a claim must be right, the TA should not be recalling it; it should be reading it.

## Where frameworks fit

Nothing above needed a framework, and for understanding agents you should build the raw loop at least once: every trace you will ever debug is this message list. Frameworks are not the wrong choice, they are the same loop plus the production concerns this notebook skipped: streaming, retries, rate-limit handling, tracing and evaluation, human approval gates, parallel tool execution, and prompt caching for the stable system prompt and schemas. Current options, roughly in order of distance from the raw loop: the tool runner built into the Anthropic SDK (`client.beta.messages.tool_runner`, which replaces `run_turn` entirely), PydanticAI for typed tools, LangChain's `create_agent` on the LangGraph runtime, demonstrated below, and smolagents for agents that act by writing code. One tier further out sit visual orchestrators such as n8n and role-scripted multi-agent frameworks such as CrewAI: prototyping surfaces and product glue rather than a different mechanism, since everything they run is still this loop. The Model Context Protocol standardises the tool boundary itself, so a transcription tool exposed once as an MCP server becomes callable from any of these and from end-user products.

## The same agent in LangChain

To make the wrapper claim concrete, here is the identical TA in LangChain. The three tool functions and the assembled system prompt transfer unchanged, the schemas are rebuilt from the same name and description strings, and `run_turn` disappears, because the loop is what the framework supplies.

In [ ]:
%pip install -q langchain langchain-anthropic

In [ ]:
from langchain.agents import create_agent
from langchain_core.tools import StructuredTool

lc_tools = [
    StructuredTool.from_function(
        func=TOOL_FUNCTIONS[t["name"]],
        name=t["name"],
        description=t["description"],
    )
    for t in TOOLS
]

login("amy")
lc_ta = create_agent(
    model="anthropic:claude-opus-5",
    tools=lc_tools,
    system_prompt=SYSTEM_PROMPT,
)

result = lc_ta.invoke(
    {"messages": [{"role": "user",
                   "content": "Check my newest take against the exercise."}]}
)
display(Markdown(result["messages"][-1].content))

Note what the framework hid: the tool calls ran, but nothing printed. The trace still exists, and inspecting it lands the central lesson of this notebook, because the framework's state is the same message list that `run_turn` maintained by hand:

In [ ]:
for m in result["messages"]:
    m.pretty_print()

What those lines bought is everything around the loop: pass `checkpointer=InMemorySaver()` and a `thread_id` and the conversation persists between `invoke` calls, swap the model string to change provider, and streaming, tracing, and middleware hooks attach without touching the agent. Within-session state is the framework's job; the learner's cross-session memory stays in our database, written at the session boundary, which is why the JSON files and not the framework remain the system of record.

## A chat interface in one file

A notebook REPL is not how a learner meets a TA. Streamlit turns the pieces above into a chat app in one Python file: the same database, the same tools, the same loop, plus rendering. The cell below writes the app next to this notebook (it reads the `studio_db/` created above), and the sidebar logs in as any of the three learners and ends the session to write memory.

In [6]:
%%writefile ta_app.py
import json, os, pathlib
from datetime import date

import anthropic
import streamlit as st
from basic_pitch.inference import predict

DB = pathlib.Path("studio_db")
MODEL = "claude-opus-5"
BPM = 100
SIXTEENTH = 60 / BPM / 4
MELODY = [60,62,64,65,67,65,64,62, 60,64,67,72,71,67,64,62,
          64,65,67,69,71,69,67,65, 64,67,71,76,74,71,67,64]
SCORE = [(i * SIXTEENTH, SIXTEENTH * 0.95, p) for i, p in enumerate(MELODY)]

PEDAGOGY = """You are a between-lessons teaching agent (TA) for one instrumental learner. \
You support their human teacher; you do not replace them.

Rules, in priority order:
1. Match the learner's age, level, and preferred feedback style, all given in the profile \
below. Constraints in teacher_notes are hard limits.
2. Tool output is evidence, not truth. Transcription misses notes and invents harmonic \
ghosts; weight consistent patterns over isolated one-note anomalies. If the report looks \
unreliable overall, say so and ask for another take.
3. Give formative feedback tied to specific bars. Lead with the single most important \
issue and mention at most one more. Never recite the full report.
4. Assign one concrete drill, sized just beyond what the memory notes say the learner \
already manages. If the same issue recurs across sessions, change strategy.
"""

@st.cache_data
def played(recording_id):
    import contextlib, io
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        _, _, events = predict(str(DB / "recordings" / f"{recording_id}.wav"))
    return sorted((round(float(s), 3), round(float(e - s), 3), int(p))
                  for s, e, p, a, b in events)

def list_recordings():
    return json.dumps([p.stem for p in
                       sorted((DB / "recordings").glob(f"{st.session_state.user}_*.wav"))])

def transcribe_recording(recording_id):
    if not recording_id.startswith(st.session_state.user):
        return json.dumps({"error": "recording belongs to another learner"})
    return json.dumps([{"onset_s": o, "duration_s": d, "midi_pitch": p}
                       for o, d, p in played(recording_id)])

def check_against_score(recording_id):
    if not recording_id.startswith(st.session_state.user):
        return json.dumps({"error": "recording belongs to another learner"})
    notes = [(o, p) for o, d, p in played(recording_id)]
    used, bars = set(), []
    for bar in range(len(SCORE) // 8):
        rows = []
        for j in range(bar * 8, (bar + 1) * 8):
            exp_onset, _, exp_pitch = SCORE[j]
            best = None
            for k, (onset, pitch) in enumerate(notes):
                if k in used or abs(onset - exp_onset) > 0.10:
                    continue
                if best is None or abs(onset - exp_onset) < abs(notes[best][0] - exp_onset):
                    best = k
            beat = 1 + (j % 8) / 4
            if best is None:
                rows.append({"beat": beat, "expected_pitch": exp_pitch, "heard_pitch": None})
            else:
                used.add(best)
                onset, pitch = notes[best]
                rows.append({"beat": beat, "expected_pitch": exp_pitch, "heard_pitch": pitch,
                             "onset_dev_ms": round((onset - exp_onset) * 1000)})
        devs = [r["onset_dev_ms"] for r in rows if r["heard_pitch"] is not None]
        bars.append({"bar": bar + 1, "notes_matched": f"{len(devs)}/8",
                     "pitch_errors": [r for r in rows
                                      if r["heard_pitch"] not in (None, r["expected_pitch"])],
                     "missed_notes": [r for r in rows if r["heard_pitch"] is None],
                     "mean_onset_dev_ms": round(sum(devs) / len(devs)) if devs else None})
    return json.dumps({"tempo_bpm_assigned": BPM, "bars": bars,
                       "extra_unmatched_notes": len(notes) - len(used)})

TOOL_FUNCTIONS = {f.__name__: f for f in
                  [list_recordings, transcribe_recording, check_against_score]}
TOOLS = [
    {"name": "list_recordings",
     "description": "List the logged-in learner's uploaded recordings by id.",
     "input_schema": {"type": "object", "properties": {}, "required": []}},
    {"name": "transcribe_recording",
     "description": "Run automatic music transcription on one recording and return raw "
                    "note events. Prefer check_against_score for graded feedback.",
     "input_schema": {"type": "object",
                      "properties": {"recording_id": {"type": "string"}},
                      "required": ["recording_id"]}},
    {"name": "check_against_score",
     "description": "Compare one recording against the assigned exercise score and return "
                    "a per-bar report of matches, pitch errors, missed notes, and mean "
                    "onset deviation in ms (positive means late).",
     "input_schema": {"type": "object",
                      "properties": {"recording_id": {"type": "string"}},
                      "required": ["recording_id"]}},
]

st.title("Practice TA")
user = st.sidebar.selectbox("Log in as", ["amy", "ben", "chloe"])
api_key = os.environ.get("ANTHROPIC_API_KEY") or st.sidebar.text_input(
    "Anthropic API key", type="password")

if st.session_state.get("user") != user:
    st.session_state.user = user
    st.session_state.chat = []
    st.session_state.messages = []

profile = json.loads((DB / "learners" / f"{user}.json").read_text())
system_prompt = (PEDAGOGY
    + "\nThe learner, their teacher's constraints, and your memory of past sessions:\n\n"
    + json.dumps(profile, indent=2, ensure_ascii=False))
st.sidebar.caption(f"{profile['name']}, {profile['level']}")

for role, text in st.session_state.chat:
    st.chat_message(role).markdown(text)

def run_turn(client):
    while True:
        response = client.messages.create(
            model=MODEL, max_tokens=4096, system=system_prompt,
            tools=TOOLS, messages=st.session_state.messages)
        st.session_state.messages.append({"role": "assistant", "content": response.content})
        if response.stop_reason != "tool_use":
            return next((b.text for b in response.content if b.type == "text"), "")
        results = []
        for block in response.content:
            if block.type == "tool_use":
                with st.status(f"{block.name}({json.dumps(block.input)})"):
                    output = TOOL_FUNCTIONS[block.name](**block.input)
                    st.code(output, language="json")
                results.append({"type": "tool_result", "tool_use_id": block.id,
                                "content": output})
        st.session_state.messages.append({"role": "user", "content": results})

if prompt := st.chat_input("Ask your TA"):
    if not api_key:
        st.error("Set ANTHROPIC_API_KEY or paste a key in the sidebar.")
        st.stop()
    st.chat_message("user").markdown(prompt)
    st.session_state.chat.append(("user", prompt))
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("assistant"):
        reply = run_turn(anthropic.Anthropic(api_key=api_key))
        st.markdown(reply)
    st.session_state.chat.append(("assistant", reply))

if st.sidebar.button("End session and save memory") and st.session_state.messages:
    client = anthropic.Anthropic(api_key=api_key)
    response = client.messages.create(
        model=MODEL, max_tokens=1024, system=system_prompt,
        messages=st.session_state.messages + [{"role": "user", "content":
            "The session is over. Write one consolidated memory note for the next "
            "session. Reply with the note text only."}])
    note = next(b.text for b in response.content if b.type == "text")
    path = DB / "learners" / f"{user}.json"
    data = json.loads(path.read_text())
    data["memory"].append({"date": date.today().isoformat(), "note": note})
    path.write_text(json.dumps(data, indent=2, ensure_ascii=False))
    st.sidebar.success("Memory saved.")

Writing ta_app.py


Run it from this notebook's folder:

```bash
pip install streamlit
streamlit run ta_app.py
```

The sidebar switches learners, tool calls unfold inside status boxes so the evidence stays inspectable, and the end-session button writes the memory note back to the same JSON files the notebook uses. Nothing agent-shaped changed; the hundred lines are rendering. A production interface would add recording upload, authentication in place of the selectbox, and streamed replies, and [](./agentic_pedagogy.md) is what should govern everything it says.

## Further reading

- *Building Effective Agents*, Anthropic's engineering guide, including when not to use a framework: <https://www.anthropic.com/engineering/building-effective-agents>
- Anthropic cookbook, runnable tool-use and memory notebooks: <https://github.com/anthropics/anthropic-cookbook>
- basic-pitch, the lightweight transcription model used here {cite}`bittner_basicpitch_2022`: <https://github.com/spotify/basic-pitch>
- Hugging Face Agents Course, an agent from scratch without a library: <https://huggingface.co/learn/agents-course>
- Model Context Protocol documentation: <https://modelcontextprotocol.io>
- LangChain agents documentation, `create_agent` and middleware: <https://docs.langchain.com/oss/python/langchain/agents>
- Streamlit chat elements, the interface primitives used above: <https://docs.streamlit.io/develop/api-reference/chat>
- Survey of LLM agents in education, the tools, memory, planning taxonomy this chapter follows {cite}`chu_llm_agents_education_2025`
- MuseAgent, tool orchestration over score and audio for one take {cite}`zhao_museagent_2026`